# Sutro · Bangla khatian text-line OCR (CRNN + CTC)

Reads one text line cropped from a **Form-5462-style khatian page** (owner names, father/village lines, dag numbers, land class, areas, shares, remarks, admin names) and returns the Bangla text.

**Data:** `khatian_form5462/ocr_cells` from the Sutro dataset — about 100k line crops (height 40 px) cut from **synthetic** pages, split **by land** so versions of one land never appear in both train and test.

**Model:** 6-layer CNN → 2-layer BiLSTM → CTC over Unicode code points (conjuncts are sequences of code points, so no special tokenizer is needed).

**Read this before quoting any number from this notebook**
- The pages are *printed-text renders* of invented data with simulated scanning, in four Bengali font families. Scores here measure how well the model reads **this synthetic distribution**. They say **nothing** about accuracy on real scanned khatians, which has not been measured.
- Split is by land, but all pages come from the same generator and the same fonts, so val/test are in-distribution.
- Metric is character error rate (CER) over Unicode code points, and exact-match accuracy per field type.

In [ ]:
import os, json, math, random, time, glob, csv, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image

SMOKE = os.environ.get("SMOKE") == "1"          # tiny CPU run to check the code
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = int(os.environ.get("EPOCHS", 2 if SMOKE else 30))
BATCH = int(os.environ.get("BATCH", 64 if SMOKE else 128))
LR = 2e-3
H = 40
RUN_ID = time.strftime("khatian-crnn-%Y%m%d-%H%M%S"); OUT = os.path.join(os.environ.get("RUN_DIR", "run"), RUN_ID); os.makedirs(OUT, exist_ok=True)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| torch", torch.__version__, "| epochs", EPOCHS, "| run", RUN_ID)

In [ ]:
# ---- locate the packed data (Kaggle input, or a local folder when run on a PC) ----
CANDIDATES = [p for p in glob.glob("/kaggle/input/**/cells_train.npz", recursive=True)] + \
             [os.path.join(os.environ.get("CELLS_DIR", "C:/land/sutro-dataset/ocr_cells_packed"), "cells_train.npz")]
ROOT = os.path.dirname(next(p for p in CANDIDATES if os.path.exists(p)))
print("data:", ROOT)

def load(split, limit=None):
    z = np.load(os.path.join(ROOT, f"cells_{split}.npz")); data, widths = z["data"], z["widths"]
    rows = list(csv.DictReader(open(os.path.join(ROOT, f"cells_{split}.csv"), encoding="utf-8")))
    offs = np.concatenate([[0], np.cumsum(widths)])
    n = len(widths) if limit is None else min(limit, len(widths))
    imgs = [data[:, offs[i]:offs[i + 1]] for i in range(n)]
    return imgs, [r["text"] for r in rows[:n]], [r["field_type"] for r in rows[:n]]

lim = dict(train=3000, val=500, test=500) if SMOKE else {}
tr_x, tr_t, tr_f = load("train", lim.get("train")); va_x, va_t, va_f = load("val", lim.get("val")); te_x, te_t, te_f = load("test", lim.get("test"))
print({k: len(v) for k, v in dict(train=tr_x, val=va_x, test=te_x).items()})
print("field types (train):", dict(collections.Counter(tr_f)))

In [ ]:
# ---- character set: every Unicode code point seen in TRAIN labels (index 0 = CTC blank) ----
chars = sorted({c for t in tr_t for c in t})
c2i = {c: i + 1 for i, c in enumerate(chars)}
print(len(chars), "symbols:", "".join(chars[:80]))
oov = sum(1 for t in va_t + te_t for c in t if c not in c2i)
print("val/test characters not in the train charset:", oov)

def keep(imgs, texts, ftypes):
    # CTC needs input length >= label length (+ repeats). Input length = width // 4.
    out = []
    for x, t, f in zip(imgs, texts, ftypes):
        need = len(t) + sum(1 for a, b in zip(t, t[1:]) if a == b)
        if x.shape[1] // 4 >= need and all(c in c2i for c in t): out.append((x, t, f))
    return out
raw = len(tr_x)
train, val, test = keep(tr_x, tr_t, tr_f), keep(va_x, va_t, va_f), keep(te_x, te_t, te_f)
print(f"kept {len(train)}/{raw} train, {len(val)} val, {len(test)} test")
# ---- run manifests: data hash, counts, and a real leakage check (split is by land) ----
import hashlib
sha = hashlib.sha256(open(os.path.join(ROOT, "cells_train.npz"), "rb").read()).hexdigest()
pages = {s_: sorted({r["page_id"] for r in csv.DictReader(open(os.path.join(ROOT, f"cells_{s_}.csv"), encoding="utf-8"))}) for s_ in ("train", "val", "test")}
lands = {s_: {p.split("-")[0] for p in v} for s_, v in pages.items()}
assert not (lands["train"] & lands["test"]) and not (lands["train"] & lands["val"]), "land leakage between splits!"
print("lands per split:", {k: len(v) for k, v in lands.items()}, "| no land overlap: OK")
json.dump(dict(source="Sutro synthetic khatian_form5462/ocr_cells (generated, invented data; 4 Bengali fonts; simulated scanning)", file="cells_train.npz", sha256=sha, height=H,
               grouping="split by land_id: every khatian version of a land is in one split", n_train=len(train), n_val=len(val), n_test=len(test), charset_size=len(chars),
               field_types_train={k: int(v) for k, v in collections.Counter(f for _, _, f in train).items()}), open(os.path.join(OUT, "dataset_manifest.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
json.dump(dict(by="land_id", pages=pages, lands={k: sorted(v) for k, v in lands.items()}), open(os.path.join(OUT, "split_manifest.json"), "w"))

In [ ]:
try:
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(8, 1, figsize=(7, 5))
    for a, (x, t, f) in zip(ax, random.sample(train, 8)):
        a.imshow(x, cmap="gray", aspect="auto"); a.set_axis_off(); a.set_title(f"{f}: {t}", fontsize=8, loc="left")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "samples.png"), dpi=110); plt.show()
except Exception as e:
    print("sample plot skipped:", repr(e)[:120])

In [ ]:
# ---- dataset with light augmentation (train only) ----
def augment(x):
    im = Image.fromarray(x)
    w, h = im.size
    im = im.resize((max(24, int(w * random.uniform(.9, 1.1))), h), Image.BILINEAR)
    im = im.rotate(random.uniform(-1.2, 1.2), resample=Image.BILINEAR, fillcolor=int(np.median(x)))
    a = np.asarray(im).astype(np.float32)
    a = (a - 128) * random.uniform(.8, 1.15) + 128 + random.uniform(-15, 15)
    a += np.random.normal(0, random.uniform(0, 5), a.shape)
    return np.clip(a, 0, 255).astype(np.uint8)

class DS(torch.utils.data.Dataset):
    def __init__(s, items, aug): s.items, s.aug = items, aug
    def __len__(s): return len(s.items)
    def __getitem__(s, i):
        x, t, f = s.items[i]
        if s.aug: x = augment(x)
        return torch.from_numpy(x.astype(np.float32) / 255.0), torch.tensor([c2i[c] for c in t], dtype=torch.long), t, f

def collate(b):
    w = max(x.shape[1] for x, *_ in b); w += (-w) % 4
    X = torch.ones(len(b), 1, H, w)
    for i, (x, *_) in enumerate(b): X[i, 0, :, :x.shape[1]] = x
    X = (X - .5) / .5
    lens = torch.tensor([x.shape[1] // 4 for x, *_ in b], dtype=torch.long)
    tg = torch.cat([y for _, y, *_ in b]); tl = torch.tensor([len(y) for _, y, *_ in b], dtype=torch.long)
    return X, lens, tg, tl, [t for *_, t, _ in b], [f for *_, f in b]

nw = 0 if (SMOKE or os.name == "nt") else 2
mk = lambda items, aug, sh: torch.utils.data.DataLoader(DS(items, aug), batch_size=BATCH, shuffle=sh, collate_fn=collate, num_workers=nw, pin_memory=DEVICE == "cuda")
dl_tr, dl_va, dl_te = mk(train, True, True), mk(val, False, False), mk(test, False, False)

In [ ]:
def conv(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class CRNN(nn.Module):
    def __init__(s, nclass):
        super().__init__()
        s.cnn = nn.Sequential(conv(1, 64), nn.MaxPool2d(2, 2), conv(64, 128), nn.MaxPool2d(2, 2), conv(128, 256), conv(256, 256), nn.MaxPool2d((2, 1), (2, 1)),
                              conv(256, 384), conv(384, 384), nn.MaxPool2d((2, 1), (2, 1)))
        s.rnn = nn.LSTM(384, 256, num_layers=2, bidirectional=True, dropout=.25)
        s.fc = nn.Linear(512, nclass)
    def forward(s, x):
        f = s.cnn(x).mean(2)                      # (B, C, T)   width // 4 time steps
        o, _ = s.rnn(f.permute(2, 0, 1))          # (T, B, 512)
        return s.fc(o).log_softmax(2)             # (T, B, V)

model = CRNN(len(chars) + 1).to(DEVICE)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")
ctc = nn.CTCLoss(blank=0, zero_infinity=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(dl_tr), pct_start=.15)
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
json.dump(dict(run_id=RUN_ID, task="Bangla khatian text-line OCR (CRNN+CTC)", model="6xconv(BN,ReLU)+2xBiLSTM(256)+linear, CTC over Unicode code points", params_m=sum(p.numel() for p in model.parameters()) / 1e6,
               epochs=EPOCHS, batch=BATCH, lr=LR, optimizer="AdamW+OneCycle", amp=DEVICE == "cuda", augmentation="width 0.9-1.1, rotate +-1.2deg, contrast/brightness jitter, gaussian noise",
               seed=SEED, device=DEVICE, framework=f"torch {torch.__version__}", smoke=SMOKE), open(os.path.join(OUT, "config.json"), "w"), indent=1)

In [ ]:
def decode(lp):                                   # greedy CTC; lp: (T, B, V)
    best = lp.argmax(2).T.cpu().numpy(); out = []
    for seq in best:
        s, prev = [], 0
        for k in seq:
            if k != prev and k != 0: s.append(chars[k - 1])
            prev = k
        out.append("".join(s))
    return out

def lev(a, b):
    d = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        p, d[0] = d[0], i
        for j, cb in enumerate(b, 1): p, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, p + (ca != cb))
    return d[-1]

@torch.no_grad()
def evaluate(dl):
    model.eval(); errs = chars_n = exact = n = 0; by = collections.defaultdict(lambda: [0, 0, 0, 0]); worst = []
    for X, lens, tg, tl, texts, fts in dl:
        preds = decode(model(X.to(DEVICE)).float())
        for p, t, f in zip(preds, texts, fts):
            e = lev(p, t); errs += e; chars_n += len(t); exact += p == t; n += 1
            b = by[f]; b[0] += e; b[1] += len(t); b[2] += p == t; b[3] += 1
            if e: worst.append((e, f, t, p))
    worst.sort(reverse=True)
    return dict(cer=errs / max(1, chars_n), exact=exact / max(1, n), n=n, by_field={k: dict(cer=v[0] / max(1, v[1]), exact=v[2] / v[3], n=v[3]) for k, v in by.items()}), worst[:15]

history = []; best_cer = 9; t0 = time.time()
for ep in range(1, EPOCHS + 1):
    model.train(); tot = k = 0
    for X, lens, tg, tl, _, _ in dl_tr:
        X = X.to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            lp = model(X)
        loss = ctc(lp.float(), tg, lens, tl)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(opt); scaler.update(); sched.step(); tot += loss.item(); k += 1
    m, _ = evaluate(dl_va); history.append(dict(epoch=ep, loss=tot / k, val_cer=m["cer"], val_exact=m["exact"]))
    flag = ""
    if m["cer"] < best_cer: best_cer = m["cer"]; torch.save(model.state_dict(), os.path.join(OUT, "best.pt")); flag = " *"
    print(f"epoch {ep:2d}/{EPOCHS}  loss {tot / k:.4f}  val CER {m['cer']:.4f}  val exact {m['exact']:.4f}  ({time.time() - t0:.0f}s){flag}")

In [ ]:
# ---- final evaluation on the held-out TEST split (best checkpoint by validation CER) ----
model.load_state_dict(torch.load(os.path.join(OUT, "best.pt"), map_location=DEVICE))
res, worst = evaluate(dl_te)
print(f"TEST  CER {res['cer']:.4f}   exact-match {res['exact']:.4f}   n={res['n']}")
print(f"{'field type':14s} {'n':>6s} {'CER':>8s} {'exact':>8s}")
for f, v in sorted(res["by_field"].items(), key=lambda kv: -kv[1]["cer"]): print(f"{f:14s} {v['n']:6d} {v['cer']:8.4f} {v['exact']:8.4f}")
print("\nworst test errors (edits, field, truth -> prediction):")
for e, f, t, p in worst: print(f"  {e:2d} {f:12s} {t!r} -> {p!r}")

In [ ]:
# ---- run artifacts: charset, history, metrics, model card, ONNX (dynamic width) + parity check ----
import importlib, subprocess, sys
for pkg in ("onnx", "onnxscript", "onnxruntime"):
    try: importlib.import_module(pkg)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
json.dump(dict(chars=chars, height=H, downsample=4, blank=0, normalize=dict(mean=.5, std=.5),
               input="float32 [B,1,40,W]: grayscale/255, pad W to a multiple of 4 with 1.0 (white), then (x-0.5)/0.5; output logprobs [T=W/4, B, V]; greedy CTC decode"),
          open(os.path.join(OUT, "charset.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
json.dump(history, open(os.path.join(OUT, "history.json"), "w"), indent=1)
model.eval(); xb, tb, fb = test[0]; inp = ((torch.from_numpy(xb.astype(np.float32) / 255.0)[None, None] - .5) / .5).to(DEVICE)
with torch.no_grad():
    for _ in range(5): model(inp)
    t_ = time.perf_counter()
    for _ in range(50): model(inp)
    ms = (time.perf_counter() - t_) * 20
json.dump(dict(run_id=RUN_ID, status="completed", task="Bangla khatian text-line OCR (synthetic Form-5462-style pages)", test=res, val_best_cer=best_cer, epochs=EPOCHS, n_train=len(train), n_val=len(val), n_test=len(test),
               inference_ms_per_line=dict(value=ms, device=DEVICE, width_px=int(xb.shape[1])), smoke=SMOKE,
               note="Synthetic printed-text distribution only; NOT a measure of accuracy on real scanned or handwritten khatians."), open(os.path.join(OUT, "metrics.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
model.cpu(); inp = inp.cpu(); onnx_path = os.path.join(OUT, "khatian_ocr.onnx"); parity = "not checked"
try:
    dummy = torch.randn(1, 1, H, 256); kw = dict(input_names=["image"], output_names=["logprobs"], dynamic_axes=dict(image={0: "b", 3: "w"}, logprobs={0: "t", 1: "b"}), opset_version=17)
    try: torch.onnx.export(model, dummy, onnx_path, dynamo=False, **kw)
    except TypeError: torch.onnx.export(model, dummy, onnx_path, **kw)
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path); a_ = model(inp).detach().numpy(); b_ = sess.run(None, dict(image=inp.numpy()))[0]
    parity = float(np.abs(a_ - b_).max()); print("onnx exported", os.path.getsize(onnx_path) // 1024, "KB | parity max|diff|", parity, "| onnx decode:", decode(torch.from_numpy(b_))[0], "| truth:", tb)
except Exception as e:
    print("ONNX export/check FAILED:", repr(e)[:300])
card = f'''# Model card — {RUN_ID}
**Task.** Read one text line cropped from a Form-5462-style Bangla khatian page (owner names, father/village lines, dag numbers, land class, areas, shares, remarks, admin names).
**Data.** Sutro synthetic `khatian_form5462/ocr_cells`: {len(train)} train / {len(val)} val / {len(test)} test lines, **split by land** (no land in two splits). All pages are font-rendered invented data (4 Bengali fonts) with simulated scanning. Not handwriting. Not real scans.
**Model.** CRNN: 6 conv blocks + 2-layer BiLSTM + CTC over {len(chars)} Unicode code points ({sum(p.numel() for p in model.parameters()) / 1e6:.2f} M params). AdamW + OneCycle, {EPOCHS} epochs, light augmentation. Selected by validation CER; evaluated once on test.
**Results (test).** CER {res['cer']:.4f}, exact-match {res['exact']:.4f} (n={res['n']}); per field type in metrics.json. Inference {ms:.1f} ms/line on {DEVICE}.
**Export.** ONNX opset 17; parity vs PyTorch max|diff| = {parity}.
**Limitations.** Synthetic printed text only: accuracy on real scanned/handwritten khatians is **not measured**. Charset limited to characters seen in training. No layout detection: lines must be cropped first. Greedy decoding; no language model.
'''
open(os.path.join(OUT, "model_card.md"), "w", encoding="utf-8").write(card)
print(sorted(os.listdir(OUT)))